# read-panther-output-sandbox
6.26.25

Doing a more systematic exploration of the Panther-output GO terms. I guess the real goal is to
compare the GO terms from Lupine and unimputed, and see which ones are unique to Lupine. 

In [1]:
import pandas as pd
import numpy as np

#### Configs

In [2]:
fdr_pval=0.05
panther_column_ids=["GO biological process complete", "reference list number", 
                    "analyzed list number", "expected", "+/-", "fold enrichment",
                    "raw pvalue", "FDR"]
brca_path="../panther_output_v1/BRCA-upreg-lupine-panther-results-v1.txt"
results_dir="../panther_output_v1/"

#### Read in the Panther output files 

In [3]:
def read_in(_cohort):
    """
    Read in and pre-process the Panther GSEA results for a single cohort,
    for both Lupine imputed and unimputed. 
    
    Parameters
    ----------
    _cohort: str, 
        The name of the CPTAC cohort 
        
    Returns
    ----------
    unimputed_df, lupine_df : `pandas`.DataFrame, 
        DataFrames for the unimputed and Lupine-imputed Panther GSEA
        results
    """
    # Read in 
    unimputed_df = pd.read_csv(results_dir+_cohort+"-upreg-lupine-panther-results.txt", 
                               sep="\t", header=None)
    lupine_df = pd.read_csv(results_dir+_cohort+"-upreg-unimputed-panther-results.txt", 
                                sep="\t", header=None)
    # Set the column IDs
    unimputed_df.columns=panther_column_ids
    lupine_df.columns=panther_column_ids

    return unimputed_df, lupine_df

def filter_GO_terms(lupine_df, unimputed_df, _fdr):
    """
    For two Panther GSEA output matrices, one for Lupine-imputed and one for
    unimputed, removes the GO terms that are not significant as per some
    (global) FDR threshold. 
    
    Parameters
    ----------
    lupine_df, unimputed_df : `pandas`.DataFrame, 
        The Lupine-imputed and unimputed Panther GSEA output matrices
    _fdr : float, 
        The FDR to use for filtering 
    
    Returns
    ----------
    lupine_df, unimputed_df : `pandas`.DataFrame,
        Filtered versions of the input dataframes 
    """
    lupine_df = lupine_df[lupine_df["FDR"] < _fdr]
    unimputed_df = unimputed_df[unimputed_df["FDR"] < _fdr]
    lupine_df = lupine_df.reset_index(drop=True)
    unimputed_df = unimputed_df.reset_index(drop=True)

    return lupine_df, unimputed_df

def get_unique_go_terms(lupine_df, unimputed_df):
    """
    Get the Panther GO terms that are unique to Lupine

    Parameters
    ----------
    lupine_df, unimputed_df : `pandas`.DataFrame, 
        The Lupine and unimputed GO term dataframes 
    
    Returns
    ----------
    lupine_df_unique : `pandas`.DataFrame, 
        Dataframe with the Lupine-only GO terms
    """
    unimputed_go_terms = unimputed_df["GO biological process complete"]
    lupine_go_terms = lupine_df["GO biological process complete"]
    shared = np.intersect1d(lupine_go_terms, unimputed_go_terms, return_indices=True)
    lupine_df_unique = lupine_df[~lupine_df.index.isin(shared[1])]

    return lupine_df_unique

#### Read in the Panther results for each cohort 

In [4]:
ccrcc_lupine, ccrcc_unimputed = read_in("CCRCC")
coad_lupine, coad_unimputed = read_in("COAD")
gbm_lupine, gbm_unimputed = read_in("GBM")
hgsc_lupine, hgsc_unimputed = read_in("HGSC")
hnscc_lupine, hnscc_unimputed = read_in("HNSCC")
lscc_lupine, lscc_unimputed = read_in("LSCC")
luad_lupine, luad_unimputed = read_in("LUAD")
pdac_lupine, pdac_unimputed = read_in("PDAC")
ucec_lupine, ucec_unimputed = read_in("UCEC")

#### Filter down to just the significant GO terms 

In [5]:
ccrcc_lupine, ccrcc_unimputed = filter_GO_terms(ccrcc_lupine, ccrcc_unimputed, fdr_pval)
coad_lupine, coad_unimputed = filter_GO_terms(coad_lupine, coad_unimputed, fdr_pval)
gbm_lupine, gbm_unimputed = filter_GO_terms(gbm_lupine, gbm_unimputed, fdr_pval)
hgsc_lupine, hgsc_unimputed = filter_GO_terms(hgsc_lupine, hgsc_unimputed, fdr_pval)
hnscc_lupine, hnscc_unimputed = filter_GO_terms(hnscc_lupine, hnscc_unimputed, fdr_pval)
lscc_lupine, lscc_unimputed = filter_GO_terms(lscc_lupine, lscc_unimputed, fdr_pval)
luad_lupine, luad_unimputed = filter_GO_terms(luad_lupine, luad_unimputed, fdr_pval)
pdac_lupine, pdac_unimputed = filter_GO_terms(pdac_lupine, pdac_unimputed, fdr_pval)
ucec_lupine, ucec_unimputed = filter_GO_terms(ucec_lupine, ucec_unimputed, fdr_pval)

#### Get BRCA
This is a unique case because there's literally nothing in the unimputed version. 
So we can just read in the Lupine GO terms. 

In [6]:
# Read in 
brca_lupine = pd.read_csv(brca_path, sep="\t")
brca_lupine.columns=panther_column_ids
# Get the significant GO terms 
brca_lupine = brca_lupine[brca_lupine["FDR"] < fdr_pval]
brca_lupine = brca_lupine.reset_index(drop=True)

#### Get the GO terms that are unique to Lupine impute

In [7]:
ccrcc_unique = get_unique_go_terms(ccrcc_lupine, ccrcc_unimputed)
coad_unique = get_unique_go_terms(coad_lupine, coad_unimputed)
gbm_unique = get_unique_go_terms(gbm_lupine, gbm_unimputed)
hgsc_unique = get_unique_go_terms(hgsc_lupine, hgsc_unimputed)
hnscc_unique = get_unique_go_terms(hnscc_lupine, hnscc_unimputed)
lscc_unique = get_unique_go_terms(lscc_lupine, lscc_unimputed)
luad_unique = get_unique_go_terms(luad_lupine, luad_unimputed)
pdac_unique = get_unique_go_terms(pdac_lupine, pdac_unimputed)
ucec_unique = get_unique_go_terms(ucec_lupine, ucec_unimputed)

print(f"BRCA # unique: {brca_lupine.shape[0]}")
print(f"CCRCC # unique: {ccrcc_unique.shape[0]}")
print(f"COAD # unique: {coad_unique.shape[0]}")
print(f"GBM # unique: {gbm_unique.shape[0]}")
print(f"HGSC # unique: {hgsc_unique.shape[0]}")
print(f"HNSCC # unique: {hnscc_unique.shape[0]}")
print(f"LSCC # unique: {lscc_unique.shape[0]}")
print(f"LUAD # unique: {luad_unique.shape[0]}")
print(f"PDAC # unique: {pdac_unique.shape[0]}")
print(f"UCEC # unique: {ucec_unique.shape[0]}")

BRCA # unique: 4
CCRCC # unique: 2
COAD # unique: 5
GBM # unique: 33
HGSC # unique: 0
HNSCC # unique: 0
LSCC # unique: 15
LUAD # unique: 1
PDAC # unique: 0
UCEC # unique: 0


#### Get a list of the Lupine-specific GO terms 

In [16]:
[x for x in luad_unique["GO biological process complete"]]

['regulation of osteoblast proliferation (GO:0033688)']

#### Get a list of the Lupine GO terms
Not necessarily Lupine-specific, could be shared with unimputed. These are still significant per our FDR threshold, though. 

In [25]:
[x for x in ucec_lupine["GO biological process complete"]]

[]